In [29]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
import zipfile

In [30]:
with zipfile.ZipFile("/content/shapes.zip", 'r') as zf:
    zf.extractall(".")

In [31]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128),
    transforms.ToTensor(),
])

In [32]:
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [33]:
train_dataset = datasets.ImageFolder("/content/shapes/train", transform=train_transform)
test_dataset = datasets.ImageFolder("/content/shapes/test", transform=test_transform)

In [34]:
class_name = train_dataset.classes
class_name

['circle', 'square', 'star', 'triangle']

In [35]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [36]:
class CheckImage(nn.Module):
   def __init__(self):
     super().__init__()
     self.first = nn.Sequential(
          nn.Conv2d(3, 32, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2),

          nn.Conv2d(32, 64, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2),

          nn.Conv2d(64, 128, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2),
          )

     self.second = nn.Sequential(
          nn.Flatten(),
          nn.Linear(128 * 16 * 16, 256),
          nn.ReLU(),
          nn.Linear(256, 6),

          )
   def forward(self, x):
     x = self.first(x)
     x = self.second(x)
     return x

In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [38]:
model = CheckImage().to(device)

In [39]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [40]:
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/15, Loss: 0.5549
Epoch 2/15, Loss: 0.2141
Epoch 3/15, Loss: 0.1482
Epoch 4/15, Loss: 0.1195
Epoch 5/15, Loss: 0.1099
Epoch 6/15, Loss: 0.1039
Epoch 7/15, Loss: 0.0917
Epoch 8/15, Loss: 0.0856
Epoch 9/15, Loss: 0.0890
Epoch 10/15, Loss: 0.0777
Epoch 11/15, Loss: 0.0735
Epoch 12/15, Loss: 0.0696
Epoch 13/15, Loss: 0.0681
Epoch 14/15, Loss: 0.0673
Epoch 15/15, Loss: 0.0699


In [41]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100*correct/total:.2f}%")

Test Accuracy: 99.84%


In [42]:
torch.save(model.state_dict(), 'model.pth')